# MX met twee gewichtsmodi: vijf-epochproef

E2M3/A32 en E2M1/A32 in **een gezamenlijk model**. FP32-mastergewichten,
gewone MX-afronding, geen truncatie. De hoge modus leert van labels; de lage
modus van losgekoppelde voorspellingen van de hoge modus. Deze notebook leest
alleen de opgeslagen rapporten: geen training, downloads of checkpointupdates.

De proef is geen paperreproductie of bewijs van eindconvergentie. Vergelijking
met de oude vijfmodusrun verandert ook activatiebeleid en trainingskosten.

In [ ]:
from pathlib import Path
RUN_DIR = Path.cwd()  # In de runkopie wordt het exacte pad ingevuld.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def load(name):
    with (RUN_DIR / name).open(encoding='utf-8') as handle:
        return json.load(handle)

report = load('run_report.json')
history = load('history.json')
metrics = pd.read_csv(RUN_DIR / 'epoch_metrics.csv')
selected = pd.read_csv(RUN_DIR / 'selected_metrics.csv')
times = pd.read_csv(RUN_DIR / 'epoch_times.csv')
selection = load('checkpoint_selection.json')
summary = load('training_summary.json')
estimate = load('timing_estimate.json')
synthetic = bool(report.get('synthetic_fixture_only', False))
if synthetic:
    display(Markdown('**SYNTHETISCHE SOFTWARETEST: geen SVHN-resultaten of bruikbare GPU-tijdschatting.**'))
assert report['schema'] == 'mx-weights-dual-pilot5-report-v1'
display(pd.DataFrame([{'status': report['status'], 'epochs': summary['completed_epochs'],
                      'optimizerstappen': summary['optimizer_steps'],
                      'train_forwardpasses': summary['training_forward_passes'],
                      'synthetisch': synthetic}]))


## Protocol en gegevenscontroles

In [ ]:
c = report['config']
display(pd.DataFrame([{'model': c['model'], 'expand': c['expand'],
    'train': c['train_examples'], 'test': c['test_examples'], 'extra': c['extra_used'],
    'activatiebits': c['activation_bits'], 'master': c['master_dtype'],
    'blokgrootte': c['mx_block_size'], 'schaal': c['mx_scale_format'],
    'distillatie': c['distillation'], 'truncatie': c['truncation']}]))
assert set(metrics.weight_bits) == {4, 6}
assert (metrics.activation_bits == 32).all()
assert len(metrics) == 4 * summary['completed_epochs']
assert not metrics.duplicated(['epoch', 'phase', 'weight_bits']).any()
assert (metrics.top1_percent - 100 * metrics.correct1 / metrics.n).abs().max() < 1e-4
assert c['distillation'] and not c['truncation'] and not c['extra_used']
assert all(r['batchnorm_counters_correct'] for r in history['epochs'])
assert all(r['gradient_checks']['finite_gradient_checks'] ==
           r['gradient_checks']['detached_teacher_checks'] ==
           r['gradient_checks']['optimizer_steps_in_phase'] for r in history['epochs'])
assert all(r['test_optimizer_steps'] == 0 for r in history['epochs'])
print('Rapportcontroles: geslaagd. Geen onafhankelijke herevaluatie van checkpoints.')
if report.get('warnings'):
    display(pd.DataFrame(report['warnings'])[['category', 'message']].drop_duplicates())
else:
    print('Geen waarschuwingen geregistreerd.')


## Nauwkeurigheid en checkpointkeuze
Beste betekent het eerste strikte maximum van **E2M3/A32** op de testset.
Beide modi komen uit datzelfde checkpoint. De testscore is niet onafhankelijk
van die selectie. Laatste wordt apart bewaard, ook als beide rollen gelijk zijn.

In [ ]:
table = selected.pivot(index=['role', 'epoch'], columns='weight_format', values='count_top1_percent')
display(table.round(4))
test = metrics[metrics.phase == 'test'].copy()
high = test[test.weight_bits == 6].sort_values('epoch')
recomputed_best = int(high.loc[high.top1_percent.idxmax(), 'epoch'])
assert recomputed_best == selection['best']['epoch']
assert selected.groupby('role').epoch.nunique().eq(1).all()
fig, ax = plt.subplots(figsize=(8, 4.5))
for bit, group in test.groupby('weight_bits', sort=False):
    ax.plot(group.epoch, group.count_top1_percent, marker='o', label=f'E2M{3 if bit == 6 else 1}/A32')
ax.set(xlabel='Epoch', ylabel='Testaccuracy (%)', title='Beide MX-modi, dezelfde testbeelden')
ax.legend(); ax.grid(True, alpha=.25); fig.tight_layout(); plt.show()


## Losses
Trainingsloss E2M3 = hard-label cross-entropy; trainingsloss E2M1 = soft-target cross-entropy.
Deze hebben niet hetzelfde leerdoel en zijn niet rechtstreeks als even grote fout
te vergelijken. **Testloss is voor beide modi hard-label cross-entropy.**

In [ ]:
display(metrics[['epoch','phase','weight_format','objective_type','objective_loss']].round(6))
fig, ax = plt.subplots(figsize=(8, 4))
for bit, group in test.groupby('weight_bits', sort=False):
    ax.plot(group.epoch, group.objective_loss, marker='o', label=f'E2M{3 if bit == 6 else 1}/A32')
ax.set(xlabel='Epoch', ylabel='Hard-label cross-entropy', title='Vergelijkbare testloss')
ax.legend(); ax.grid(True, alpha=.25); fig.tight_layout(); plt.show()


## Tijd en updates
Tijd omvat deze geinstrumenteerde emulator, geen native MX-hardware.
Een honderd-epochraming uit vijf epochs is geen gegarandeerde looptijd.

In [ ]:
display(times[['epoch','train_seconds','test_seconds','checkpoint_seconds','other_seconds','total_epoch_seconds']].round(3))
print(f"Gemeten epochtijd samen: {times.total_epoch_seconds.sum():.2f} s")
if estimate.get('available'):
    print(f"Raming 100 epochs: {estimate['estimated_100_epoch_seconds']/60:.1f} minuten")
    print('Exclusief eerste datasetvoorbereiding en eindrapportage; spreidingsband is geen betrouwbaarheidsinterval.')
updates = pd.read_csv(RUN_DIR / 'weight_updates.csv')
display(updates.groupby(['epoch','weight_bits'])[
    ['master_changed_values','changed_element_codes_including_padding','changed_scale_codes']].sum())
print('Masterupdates zijn dezelfde gedeelde tensors; tel de twee modustotalen NIET bij elkaar op.')
print('Initialisatiehash:', summary['initial_model_sha256'])


## Exports en interpretatie
Per geselecteerd checkpoint worden **twee aparte direct afgeronde MX-exports** gemaakt.
E2M1 komt nog niet uit getrunceerde E2M3-codes. Dit is dus nog geen bewijs dat een opgeslagen
hoogste MX-model volstaat voor beide precisies. De exportcontrole vergelijkt gereconstrueerde
gewichten; deze trainingsrunner doet geen extra volledige testpass vanuit de exports.

In [ ]:
export_rows=[]
for role, modes in report['mx_exports'].items():
    for bit, details in modes.items():
        export_rows.append({'role': role, 'bits': bit, 'epoch': details['epoch'],
                           'bestand_bytes': details['bytes'], 'gewichten_exact': details['decoded_weights_exact'],
                           'zonder_6_masters': details['six_fp32_masters_absent'],
                           'test_opnieuw': details['full_test_forward_rerun']})
display(pd.DataFrame(export_rows))
print('Deel feedback_bundle.zip uit:', RUN_DIR)
